# Naive Bayes for SMS Spam Classification

**CS156 - Finding Patterns in Data with Machine Learning**

This notebook organizes the Session 5 PCW into a reusable walkthrough of:

- joint, marginal, and conditional probability,
- conditional independence and why Naive Bayes is "naive",
- latent / unobserved class variables,
- TF-IDF text representations,
- Multinomial Naive Bayes,
- learned parameters,
- geometric decision boundaries,
- confusion matrices,
- feature probabilities,
- Laplace smoothing,
- interpreting which tokens push a prediction toward spam or ham.

The goal is to connect the probability formulas to the actual attributes learned by scikit-learn.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

## 1. Probability foundations

### Joint probability

A joint probability describes multiple events together:

$$
P(A,B)=P(A\cap B).
$$

### Marginal probability

A marginal probability describes one variable after summing / integrating over the others:

$$
P(A)=\sum_b P(A,b).
$$

### Conditional probability

A conditional probability asks how likely one event is after another is known:

$$
P(A\mid B)
=
\frac{P(A,B)}{P(B)}.
$$

### Independence

If $A$ and $B$ are independent,

$$
P(A,B)=P(A)P(B).
$$

**Important correction for study:** independence does not mean that the events have "no intersection." Independent events can occur together; independence means that observing one does not change the probability of the other.

## 2. What makes Naive Bayes "naive"?

Naive Bayes assumes the features are **conditionally independent given the class**.

For a message represented by features $x_1,\ldots,x_d$ and a class $c$,

$$
P(x_1,\ldots,x_d\mid c)
\approx
\prod_{i=1}^d P(x_i\mid c).
$$

For SMS classification, the class might be `spam` or `ham`.

The assumption is strong because real words are related. For example, `"New"` and `"York"` are not truly independent linguistic features.

A separate modeling choice also matters here: the default unigram `TfidfVectorizer` represents a document as token features and does not encode ordinary word order. That loss of order comes from the text representation; the "naive" part of Naive Bayes is specifically the conditional-independence assumption.

### When can the assumption break down?

If two features carry nearly the same information, treating them as independent can effectively count similar evidence multiple times.

The model can still classify well even when the assumption is imperfect because accurate probability modeling is not always necessary for a useful decision boundary.

### Latent / unobserved variables

A latent variable is not directly observed but is inferred from observed data.

In supervised training, the spam/ham label is observed. For a **new** SMS, however, the class is unknown and must be inferred from its observed features.

## 3. Bayes classification

Bayes' rule gives

$$
P(c\mid x)
=
\frac{P(x\mid c)P(c)}{P(x)}.
$$

For classification, $P(x)$ is the same for every candidate class, so we can compare

$$
P(c\mid x)
\propto
P(c)P(x\mid c).
$$

Under the naive conditional-independence assumption,

$$
P(c\mid x)
\propto
P(c)\prod_i P(x_i\mid c).
$$

Computers usually work in log space:

$$
\log \operatorname{score}(c)
=
\log P(c)
+
\sum_i x_i\log P(X_i\mid c).
$$

The product becomes a sum, which is numerically more stable and exposes the geometry of the classifier.

## 4. Load the SMS dataset

The dataset contains SMS messages labeled `ham` or `spam`.

The notebook follows the PCW's source dataset and uses the text in column `v2` and labels in `v1`.

In [ ]:
SMS_URL = (
    "https://raw.githubusercontent.com/"
    "milindsoorya/Spam-Classifier-in-python/main/dataset/spam.csv"
)

sms_df = pd.read_csv(SMS_URL, encoding="latin-1")

labels = sms_df["v1"]
texts = sms_df["v2"]

print(labels.value_counts())
texts.head(3)

## 5. TF-IDF: converting text into vectors

`TfidfVectorizer` first builds a vocabulary. Every vocabulary token becomes one feature dimension.

For token $t$ in document $d$, the basic idea is

$$
\operatorname{TFIDF}(t,d)
=
\operatorname{TF}(t,d)
\operatorname{IDF}(t).
$$

Term frequency rewards words that appear in the current document. Inverse document frequency reduces the weight of words that occur in many documents.

With scikit-learn's defaults, IDF is smoothed and each output row is also L2-normalized.

Because a message contains only a small fraction of the full vocabulary, the result is stored as a **sparse matrix**.

In [ ]:
vectorizer = TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)

print("Matrix shape:", vec_texts.shape)
print("Number of vocabulary tokens:", len(vectorizer.vocabulary_))

# Show only the nonzero values for the first three messages.
print(vec_texts[:3])

For this dataset, the PCW produced a matrix with shape `(5572, 8672)`:

- 5,572 rows = messages,
- 8,672 columns = vocabulary features.

A sparse entry such as `(0, 8267) = 0.182...` means that feature 8267 has that TF-IDF weight in message 0.

`vectorizer.vocabulary_` maps tokens to those column indices.

In [ ]:
vocab_preview = sorted(
    vectorizer.vocabulary_.items(),
    key=lambda item: item[1],
)[:20]

vocab_preview

## 6. Train Multinomial Naive Bayes

`MultinomialNB` estimates one feature distribution per class.

For each token feature $i$, the model learns quantities corresponding to

$$
P(X_i\mid \text{ham})
\qquad\text{and}\qquad
P(X_i\mid \text{spam}),
$$

stored internally as log probabilities.

Scikit-learn's Multinomial Naive Bayes accepts nonnegative fractional inputs such as TF-IDF values, even though the classic multinomial model is often introduced with integer token counts.

In [ ]:
model = MultinomialNB()
model.fit(vec_texts, labels)

pred_labels = model.predict(vec_texts)

print("Classes:", model.classes_)
print("feature_log_prob_ shape:", model.feature_log_prob_.shape)
print("class_log_prior_ shape:", model.class_log_prior_.shape)

## 7. How many parameters are learned?

There are

$$
8672
$$

input features and two classes.

The main feature-probability table therefore contains

$$
2\times8672
=
\boxed{17{,}344}
$$

feature parameters.

In addition, scikit-learn stores two class log-priors, one for ham and one for spam.

So:

- **17,344 feature/class parameters** is the answer emphasized by the PCW;
- counting the two class-prior values as well gives **17,346 learned scalar probability parameters**.

The input space is still 8,672-dimensional. The 17,344 value counts two different coefficient vectors operating on the same 8,672 coordinates.

## 8. Geometry: why the decision boundary is linear

For two classes, prediction compares two log scores:

$$
S_{\text{spam}}(x)
=
b_s+w_s^Tx
$$

and

$$
S_{\text{ham}}(x)
=
b_h+w_h^Tx.
$$

The decision boundary occurs where they are equal:

$$
b_s+w_s^Tx
=
b_h+w_h^Tx.
$$

Rearranging,

$$
(w_s-w_h)^Tx+(b_s-b_h)=0.
$$

This is a linear equation in $x$.

- in 2 dimensions: a straight line,
- in 3 dimensions: a plane,
- in 8,672 dimensions: a hyperplane.

Unlike a decision tree, Naive Bayes does not recursively divide the space into axis-aligned boxes.

## 9. Confusion matrix

The original PCW predicts on the same messages used for training. This reproduces that result, but it should be interpreted as **training performance**, not as an estimate of generalization to new messages.

In [ ]:
confusion_mat = metrics.confusion_matrix(
    labels,
    pred_labels,
    labels=["ham", "spam"],
)

sns.heatmap(
    confusion_mat,
    square=True,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=["ham", "spam"],
    yticklabels=["ham", "spam"],
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Training confusion matrix")
plt.show()

accuracy = metrics.accuracy_score(labels, pred_labels)
precision = metrics.precision_score(
    labels,
    pred_labels,
    pos_label="spam",
)
recall = metrics.recall_score(
    labels,
    pred_labels,
    pos_label="spam",
)

print("Training accuracy:", accuracy)
print("Spam precision:", precision)
print("Spam recall:", recall)

The PCW confusion matrix was:

\[
\begin{bmatrix}
4825 & 0 \\
132 & 615
\end{bmatrix}
\]

when rows are true labels and columns are predicted labels in the order `[ham, spam]`.

That corresponds to about **97.6% training accuracy**. It made no ham→spam errors in this training evaluation, but missed 132 spam messages.

The operational importance of those errors depends on the application. In a phishing-like setting, false negatives can be particularly costly.

**Evaluation caveat:** because the same data are used for fitting and prediction, a train/test split or cross-validation is needed for a meaningful estimate of performance on unseen messages.

## 10. Inspect the learned feature probabilities

`feature_log_prob_` has shape

$$
(2,8672).
$$

Each row corresponds to a class and each column corresponds to the same token index used by the vectorizer.

Exponentiating converts the stored log values back to probabilities.

In [ ]:
tokens = np.array(
    [
        token
        for token, index in sorted(
            vectorizer.vocabulary_.items(),
            key=lambda item: item[1],
        )
    ]
)

spam_idx = list(model.classes_).index("spam")
ham_idx = list(model.classes_).index("ham")

spam_probs = np.exp(model.feature_log_prob_[spam_idx])
ham_probs = np.exp(model.feature_log_prob_[ham_idx])

print(tokens.shape)
print(spam_probs.shape)

### 30 tokens with the highest \(P(X_i \mid spam)\)

In [ ]:
top_indices = np.argsort(spam_probs)[-30:][::-1]

top_tokens = tokens[top_indices]
top_probs = spam_probs[top_indices]

plt.figure(figsize=(14, 6))
plt.bar(top_tokens, top_probs)
plt.xticks(rotation=90)
plt.xlabel("Token")
plt.ylabel("P(X | spam)")
plt.title("30 tokens with highest P(X | spam)")
plt.show()

### 30 tokens with the lowest \(P(X_i \mid spam)\)

Many low-probability tokens are tied. A horizontal chart makes the labels easier to read, but equal learned probabilities will still produce equal-length bars.

In [ ]:
bottom_indices = np.argsort(spam_probs)[:30]

bottom_tokens = tokens[bottom_indices]
bottom_probs = spam_probs[bottom_indices]

minimum = spam_probs.min()
number_tied = np.sum(np.isclose(spam_probs, minimum))

print("Minimum probability:", minimum)
print("Number of tokens tied at minimum:", number_tied)

plt.figure(figsize=(9, 9))
plt.barh(bottom_tokens, bottom_probs)
plt.xlabel("P(X | spam)")
plt.ylabel("Token")
plt.title("30 tokens with lowest P(X | spam)")
plt.gca().invert_yaxis()
plt.ticklabel_format(axis="x", style="scientific", scilimits=(0, 0))
plt.show()

## 11. Why are so many minimum probabilities identical?

`MultinomialNB` uses additive smoothing by default with

$$
\alpha=1.
$$

Conceptually, a class-conditional feature probability is estimated from a smoothed feature count:

$$
P(X_i\mid c)
=
\frac{N_{ci}+\alpha}
{\sum_j N_{cj}+\alpha d}.
$$

If many features have zero accumulated feature mass in spam, then all of them have the same numerator:

$$
0+\alpha.
$$

Because they also share the same denominator, they receive the same minimum probability.

In the PCW run, **5,805 tokens were tied at the minimum spam probability**.

## 12. What makes a token push toward spam or ham?

A high value of

$$
P(X_i\mid spam)
$$

alone does not necessarily mean a token is highly discriminative. A token might also be common in ham.

The useful comparison is the difference in class log-probabilities:

$$
\log P(X_i\mid spam)
-
\log P(X_i\mid ham).
$$

- positive values push the log score toward spam,
- negative values push it toward ham,
- values near zero provide little class-specific evidence.

In [ ]:
log_odds = (
    model.feature_log_prob_[spam_idx]
    - model.feature_log_prob_[ham_idx]
)

spam_associated = np.argsort(log_odds)[-20:][::-1]
ham_associated = np.argsort(log_odds)[:20]

print("Most spam-associated tokens:")
print(list(zip(tokens[spam_associated], log_odds[spam_associated])))

print("\nMost ham-associated tokens:")
print(list(zip(tokens[ham_associated], log_odds[ham_associated])))

## 13. Main takeaways

1. Naive Bayes assumes **conditional independence of features given the class**.
2. TF-IDF converts each SMS into an 8,672-dimensional nonnegative feature vector.
3. Multinomial Naive Bayes learns one feature distribution for ham and one for spam.
4. The main feature table therefore has 17,344 entries, but each message still lives in an 8,672-dimensional space.
5. Taking logs converts the Naive Bayes product into a weighted sum, giving a linear decision boundary in feature space.
6. Additive smoothing prevents unseen class-feature combinations from receiving zero probability.
7. The most discriminative tokens are identified by comparing their class-conditional probabilities, not by looking at spam probability alone.
8. The original confusion matrix is a training-set result; unseen-data evaluation requires a holdout set or cross-validation.